Plan:

Evaluate the previously trained nnU-Net 2D and CR-Semi-derived 2D models on the same 61 validation images used for Gemini and BiomedParse.

The evaluation includes:
1. Locating existing predictions and model checkpoints.
2. Generating segmentation predictions if necessary.
3. Extracting connected components and geometric measurements.
4. Preparing figures for manual visual assessment.
5. Comparing segmentation-based and VLM-based annotation methods.

BiomedParse pseudo-labels are not considered expert ground truth

In [ ]:
# Mount Google Drive and define project paths.

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

VLM_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

BIOMEDPARSE_DIR = Path(
    "/content/drive/MyDrive/biomedparse_project"
)

NNUNET_RAW_DIR = (
    BIOMEDPARSE_DIR / "nnunet_raw" / "Dataset001_Follicles"
)

MANIFEST_PATH = VLM_DIR / "image_manifest.csv"

print("VLM project:", VLM_DIR.exists())
print("BiomedParse project:", BIOMEDPARSE_DIR.exists())
print("Dataset:", NNUNET_RAW_DIR.exists())
print("Manifest:", MANIFEST_PATH.exists())

Mount Google Drive and define project paths

In [ ]:
# Locate existing model checkpoints and segmentation predictions.

checkpoint_extensions = {".pth", ".pt", ".ckpt", ".pkl"}
prediction_extensions = {".nii", ".nii.gz", ".png", ".npz"}

checkpoint_files = []
candidate_prediction_dirs = []

for path in BIOMEDPARSE_DIR.rglob("*"):

    if not path.is_file():
        continue

    name = path.name.lower()
    path_text = str(path).lower()

    # Find model checkpoint files.
    if (
        path.suffix.lower() in checkpoint_extensions
        or "checkpoint" in name
        or "model_best" in name
    ):
        checkpoint_files.append(path)

    # Find possible model prediction directories.
    if any(keyword in path_text for keyword in [
        "prediction",
        "inference",
        "nnunet_results",
        "cr_semi",
        "cr-semi",
    ]):
        if (
            path.suffix.lower() in prediction_extensions
            or name.endswith(".nii.gz")
        ):
            candidate_prediction_dirs.append(path.parent)

print("CHECKPOINT CANDIDATES:")
for path in checkpoint_files[:40]:
    print(path)

print("\nPOSSIBLE PREDICTION DIRECTORIES:")
for directory in sorted(set(candidate_prediction_dirs))[:40]:
    print(directory)

print("\nTotal checkpoint candidates:", len(checkpoint_files))
print("Total candidate directories:", len(set(candidate_prediction_dirs)))

Locate existing model checkpoints and segmentation predictions